In [1]:
# Importing ChexFound Code Base

import os
import sys
import importlib

# 1. Setup repository path
repo_dir = '/kaggle/working/CheXFound'
if not os.path.exists(os.path.join(repo_dir, 'chexfound', '__init__.py')):
    print("Cloning CheXFound codebase...")
    !rm -rf /kaggle/working/CheXFound
    !git clone https://github.com/SuyogMagar/CheXFound.git /kaggle/working/CheXFound

# 2. Register path permanently for this notebook session
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

importlib.invalidate_caches()

# 3. Test import to guarantee future cells will not fail
from chexfound.models import vision_transformer as vits
print("\n✓ SUCCESS: CheXFound codebase is imported, verified, and locked in sys.path.")

Cloning CheXFound codebase...
Cloning into '/kaggle/working/CheXFound'...
remote: Enumerating objects: 460, done.
remote: Counting objects: 100% (460/460), done.
remote: Compressing objects: 100% (358/358), done.
remote: Total 460 (delta 134), reused 407 (delta 96), pack-reused 0 (from 0)
Receiving objects: 100% (460/460), 24.14 MiB | 33.95 MiB/s, done.
Resolving deltas: 100% (134/134), done.

✓ SUCCESS: CheXFound codebase is imported, verified, and locked in sys.path.


/kaggle/working/CheXFound/chexfound/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/kaggle/working/CheXFound/chexfound/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/kaggle/working/CheXFound/chexfound/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [ ]:
# Load and Freeze the ChexFound Trained Parameters

import os
import glob
import torch
import torch.nn as nn
from chexfound.models import vision_transformer as vits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device for inference: {device}")

# 1. Locate teacher checkpoint
teacher_files = glob.glob('/kaggle/input/**/teacher_checkpoint.pth', recursive=True)
assert len(teacher_files) > 0, "Could not find teacher_checkpoint.pth in /kaggle/input/"
teacher_path = teacher_files[0]
print(f"Loading weights from: {teacher_path}")

# 2. Instantiate ViT-Large backbone (512x512 resolution, patch_size=16)
model = vits.vit_large(patch_size=16, img_size=512)

# 3. Load checkpoint and align state dict keys
teacher_ckpt = torch.load(teacher_path, map_location='cpu')
state_dict = teacher_ckpt.get('teacher', teacher_ckpt)
clean_state_dict = {
    (k[9:] if k.startswith('backbone.') else k): v 
    for k, v in state_dict.items()
}

missing_keys, unexpected_keys = model.load_state_dict(clean_state_dict, strict=False)

# 4. Freeze backbone and mount on GPU
model.eval()
model.to(device)
for param in model.parameters():
    param.requires_grad = False

print(f"\n✓ CheXFound ViT-Large Loaded:")
print(f"  - Missing keys: {len(missing_keys)}")
print(f"  - Unexpected keys: {len(unexpected_keys)}")
print(f"  - All parameters frozen: {all(not p.requires_grad for p in model.parameters())}")

# 5. Sanity check: Run a dummy forward pass through the last 4 blocks
dummy = torch.randn(1, 3, 512, 512).to(device)
with torch.no_grad():
    sample_layers = model.get_intermediate_layers(dummy, n=4, return_class_token=True)
    # Each block outputs: (patch_tokens [B, 1024, 1024], cls_token [B, 1024])
    patch_sample, cls_sample = sample_layers[-1]
    
print(f"\n✓ Forward check successful:")
print(f"  - Patch tokens shape: {patch_sample.shape} (1024 spatial patches)")
print(f"  - CLS token shape   : {cls_sample.shape} (1024 dimensions)")

In [ ]:
# Full End-to-End Production Script: VinDr-CXR (Table IV Benchmarks)
# Memory-Safe, Strict 15,000 Cohort Filtering, Flat Serialization, & Benchmark Evaluation

import gc
import glob
import importlib
import os
import re
import subprocess
import sys
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
from torchvision.transforms import InterpolationMode
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# 1. Setup CheXFound Repository
repo_dir = '/kaggle/working/CheXFound'
if not os.path.exists(os.path.join(repo_dir, 'chexfound', '__init__.py')):
  subprocess.run(['rm', '-rf', repo_dir], check=False)
  subprocess.run(
      [
          'git',
          'clone',
          'https://github.com/RPIDIAL/CheXFound.git',
          repo_dir,
      ],
      check=False,
  )
  if not os.path.exists(os.path.join(repo_dir, 'chexfound', '__init__.py')):
    subprocess.run(
        [
            'git',
            'clone',
            'https://github.com/SuyogMagar/CheXFound.git',
            repo_dir,
        ],
        check=True,
    )

if repo_dir not in sys.path:
  sys.path.insert(0, repo_dir)

importlib.invalidate_caches()
from chexfound.models import vision_transformer as vits

# 2. Target 6 Pathologies from Table IV (Yang et al.)
TARGET_CLASSES = [
    'Aortic enlargement',
    'Cardiomegaly',
    'Pulmonary fibrosis',
    'Pleural thickening',
    'Pleural effusion',
    'Pneumothorax',
]

save_dir = '/kaggle/working/vindr_safe_cache'
os.makedirs(save_dir, exist_ok=True)
p_x_linear = os.path.join(save_dir, 'X_linear.pt')
p_coarse = os.path.join(save_dir, 'coarse_patches.pt')
p_cls = os.path.join(save_dir, 'last_block_cls.pt')
p_y = os.path.join(save_dir, 'Y_all.pt')

# 3. Check for Existing Cache or Run Feature Extraction
if (
    os.path.exists(p_x_linear)
    and os.path.exists(p_coarse)
    and os.path.exists(p_cls)
    and os.path.exists(p_y)
):
  print(f'✓ Found existing safe cache in {save_dir}/. Loading...')
  X_linear = torch.load(p_x_linear, map_location='cpu')
  coarse_patches = torch.load(p_coarse, map_location='cpu')
  last_block_cls = torch.load(p_cls, map_location='cpu')
  Y_all = torch.load(p_y, map_location='cpu')
  Y_np = Y_all.numpy()
  n_total = len(Y_all)
  print(f'✓ Loaded {n_total} validated VinDr scans cleanly.')
else:
  print('Searching for VinDr-CXR annotations and images...')
  vin_csvs = [
      p
      for p in glob.glob('/kaggle/input/**/train.csv', recursive=True)
      if 'vin' in p.lower()
  ]
  assert (
      len(vin_csvs) > 0
  ), 'train.csv not found! Ensure awsaf49/vinbigdata-512-image-dataset is added.'
  annot_df = pd.read_csv(vin_csvs[0])
  id_col = 'image_id' if 'image_id' in annot_df.columns else annot_df.columns[0]
  class_col = 'class_name' if 'class_name' in annot_df.columns else 'finding'

  # Strict isolation: Only load image files from VinBigData / VinDr folders
  all_imgs = {}
  for p in (
      glob.glob('/kaggle/input/**/*.png', recursive=True)
      + glob.glob('/kaggle/input/**/*.jpg', recursive=True)
  ):
    p_low = p.lower()
    if '__macosx' in p_low:
      continue
    if 'vinbigdata' in p_low or 'vindr' in p_low:
      img_id = os.path.splitext(os.path.basename(p))[0]
      all_imgs[img_id] = p

  # Strictly filter to the 15,000 labelled cohort present in train.csv
  labelled_ids = set(annot_df[id_col].unique())
  filtered_imgs = {k: v for k, v in all_imgs.items() if k in labelled_ids}
  print(
      f'✓ Strictly isolated labelled VinDr images: {len(filtered_imgs)}'
      ' (Expected: 15,000)'
  )

  valid_records = annot_df[annot_df[class_col].isin(TARGET_CLASSES)]
  image_abnormalities = (
      valid_records.groupby(id_col)[class_col].apply(set).to_dict()
  )

  records = []
  for img_id, p in filtered_imgs.items():
    findings = image_abnormalities.get(img_id, set())
    labels = [1 if c in findings else 0 for c in TARGET_CLASSES]
    records.append({'image_id': img_id, 'path': p, 'labels': labels})

  df_vindr = (
      pd.DataFrame(records).sort_values('image_id').reset_index(drop=True)
  )
  Y_all = torch.tensor(
      np.array(df_vindr['labels'].tolist()), dtype=torch.float32
  )
  Y_np = Y_all.numpy()
  n_total = len(df_vindr)
  print(f'✓ Total Cohort Verified: {n_total} scans')

  # Load Backbone Weights
  teacher_files = glob.glob(
      '/kaggle/input/**/teacher_checkpoint.pth', recursive=True
  )
  assert len(teacher_files) > 0, 'teacher_checkpoint.pth missing!'
  teacher_ckpt = torch.load(teacher_files[0], map_location='cpu')
  raw_state = teacher_ckpt.get('teacher', teacher_ckpt)

  clean_state = {}
  for k, v in raw_state.items():
    if k.startswith('dino_head.') or k.startswith('ibot_head.'):
      continue
    new_k = k
    for pfx in ['module.', 'backbone.']:
      if new_k.startswith(pfx):
        new_k = new_k[len(pfx) :]
    new_k = re.sub(r'^blocks\.\d+\.(\d+)\.', r'blocks.\1.', new_k)
    clean_state[new_k] = v

  has_swiglu = any('w12' in k for k in clean_state.keys())
  has_layerscale = any('ls1' in k for k in clean_state.keys())
  has_registers = 'register_tokens' in clean_state
  num_registers = clean_state['register_tokens'].shape[1] if has_registers else 0
  pos_shape = clean_state['pos_embed'].shape
  native_res = int(((pos_shape[1] - 1) ** 0.5) * 16)

  model = vits.vit_large(
      patch_size=16,
      img_size=native_res,
      init_values=1e-5 if has_layerscale else None,
      ffn_layer='swiglu' if has_swiglu else 'mlp',
      block_chunks=0,
      num_register_tokens=num_registers,
  )
  msg = model.load_state_dict(clean_state, strict=False)
  assert (
      len(msg.missing_keys) == 0
  ), f'Backbone missing keys: {msg.missing_keys}'
  print('✓ CheXFound ViT-Large backbone loaded (0 missing keys)')

  model.eval().to(device)
  for p in model.parameters():
    p.requires_grad = False

  eval_transform = T.Compose([
      T.Resize((512, 512), interpolation=InterpolationMode.BICUBIC),
      T.ToTensor(),
      T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
  ])

  class VinDrDataset(Dataset):

    def __init__(self, df, transform):
      self.df = df
      self.transform = transform

    def __len__(self):
      return len(self.df)

    def __getitem__(self, idx):
      img = Image.open(self.df.iloc[idx]['path']).convert('RGB')
      return self.transform(img), idx

  loader = DataLoader(
      VinDrDataset(df_vindr, eval_transform),
      batch_size=32,
      shuffle=False,
      num_workers=2,
  )

  def pool_pyramid_tokens(patch_tokens):
    B, N, D = patch_tokens.shape
    H = W = int(N**0.5)
    x = patch_tokens.transpose(1, 2).view(B, D, H, W)
    p8 = F.adaptive_avg_pool2d(x, (4, 4)).flatten(2).transpose(1, 2)
    p4 = F.adaptive_avg_pool2d(x, (8, 8)).flatten(2).transpose(1, 2)
    p2 = F.adaptive_avg_pool2d(x, (16, 16)).flatten(2).transpose(1, 2)
    return torch.cat([p8, p4, p2], dim=1)  # 16 + 64 + 256 = 336 tokens

  X_linear = torch.zeros((n_total, 8192), dtype=torch.float32)
  coarse_patches = torch.zeros((n_total, 336, 1024), dtype=torch.float16)
  last_block_cls = torch.zeros((n_total, 1024), dtype=torch.float32)

  ptr = 0
  with torch.no_grad():
    for imgs, _ in tqdm(loader, desc='Extracting VinDr (15,000 Scans)'):
      b_size = imgs.size(0)
      imgs = imgs.to(device)
      layers = model.get_intermediate_layers(imgs, n=4, return_class_token=True)

      cls_all = torch.cat([c for _, c in layers], dim=-1).cpu()
      avg_all = torch.cat([p.mean(dim=1) for p, _ in layers], dim=-1).cpu()
      X_linear[ptr : ptr + b_size] = torch.cat([cls_all, avg_all], dim=-1)

      p_last, c_last = layers[-1]
      last_block_cls[ptr : ptr + b_size] = c_last.cpu()
      coarse_patches[ptr : ptr + b_size] = (
          pool_pyramid_tokens(p_last).cpu().half()
      )
      ptr += b_size

  del model
  torch.cuda.empty_cache()
  gc.collect()

  # Save individually using raw flat format to avoid zipfile buffer errors
  torch.save(X_linear, p_x_linear, _use_new_zipfile_serialization=False)
  torch.save(coarse_patches, p_coarse, _use_new_zipfile_serialization=False)
  torch.save(last_block_cls, p_cls, _use_new_zipfile_serialization=False)
  torch.save(Y_all, p_y, _use_new_zipfile_serialization=False)
  print(f'✓ Successfully generated and saved safe cache to {save_dir}/')

# 4. Stratified 70:10:20 Partition (10,500 Train, 1,500 Val, 3,000 Test)
indices = np.arange(len(Y_np))
has_disease = (Y_np.sum(axis=1) > 0).astype(int)
train_val_idx, test_idx = train_test_split(
    indices, test_size=0.20, random_state=42, stratify=has_disease
)
train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=0.125,
    random_state=42,
    stratify=has_disease[train_val_idx],
)

print(
    f'\nSplit Sizes -> Train: {len(train_idx)} | Val: {len(val_idx)} | Test:'
    f' {len(test_idx)}'
)

# =========================================================================
# BENCHMARK 1: LINEAR PROBE (Multi-Label 8192-dim)
# =========================================================================
X_norm = F.normalize(X_linear, p=2, dim=-1).numpy()
X_tr, Y_tr = X_norm[train_idx], Y_np[train_idx]
X_va, Y_va = X_norm[val_idx], Y_np[val_idx]
X_te, Y_te = X_norm[test_idx], Y_np[test_idx]

lp_aurocs, lp_auprcs = [], []
for c_idx, c_name in enumerate(TARGET_CLASSES):
  y_tr_c, y_va_c, y_te_c = Y_tr[:, c_idx], Y_va[:, c_idx], Y_te[:, c_idx]

  best_c, best_val_auc = 1.0, -1.0
  for c_val in [0.001, 0.01, 0.1, 1.0, 5.0, 10.0]:
    clf = LogisticRegression(C=c_val, max_iter=1000, solver='lbfgs')
    clf.fit(X_tr, y_tr_c)
    v_auc = roc_auc_score(y_va_c, clf.predict_proba(X_va)[:, 1])
    if v_auc > best_val_auc:
      best_val_auc = v_auc
      best_c = c_val

  X_tr_val = np.vstack([X_tr, X_va])
  y_tr_val_c = np.concatenate([y_tr_c, y_va_c])
  final_clf = LogisticRegression(C=best_c, max_iter=1000, solver='lbfgs')
  final_clf.fit(X_tr_val, y_tr_val_c)

  probs = final_clf.predict_proba(X_te)[:, 1]
  lp_aurocs.append(roc_auc_score(y_te_c, probs))
  lp_auprcs.append(average_precision_score(y_te_c, probs))

mean_lp_auroc = np.mean(lp_aurocs)
mean_lp_auprc = np.mean(lp_auprcs)

print('\n' + '=' * 70)
print('     BENCHMARK 1: LINEAR PROBE (VINDR-CXR)')
print('=' * 70)
for i, c in enumerate(TARGET_CLASSES):
  print(f'{c:22} | AUROC: {lp_aurocs[i]:.4f} | AUPRC: {lp_auprcs[i]:.4f}')
print('-' * 70)
print(
    f'Mean Test AUPRC : {mean_lp_auprc:.4f}  (Table IV Target: 0.298 [0.280 -'
    ' 0.317])'
)
print(
    f'Mean Test AUROC : {mean_lp_auroc:.4f}  (Table IV Target: 0.838 [0.776 -'
    ' 0.866])'
)
print('=' * 70)


# =========================================================================
# BENCHMARK 2: GLoRI MULTI-QUERY HEAD
# =========================================================================
class CompactGLoRIHead(nn.Module):

  def __init__(self, in_dim=1024, glori_dim=768, num_queries=6):
    super().__init__()
    self.num_queries = num_queries
    self.disease_queries = nn.Parameter(
        torch.randn(1, num_queries, glori_dim) * 0.02
    )
    self.pyramid_proj = nn.Sequential(
        nn.Linear(in_dim, glori_dim), nn.ReLU(), nn.LayerNorm(glori_dim)
    )
    self.temp_mlp = nn.Sequential(
      nn.Linear(glori_dim, glori_dim // 2),
      nn.Tanh(),
      nn.Linear(glori_dim // 2, num_queries),
    )
    self.mha_coarse = nn.MultiheadAttention(
        embed_dim=glori_dim, num_heads=8, batch_first=True
    )
    self.cls_proj = nn.Sequential(
        nn.Linear(in_dim, glori_dim), nn.LayerNorm(glori_dim)
    )
    self.norm = nn.LayerNorm(glori_dim * 2)
    self.classifier = nn.Linear(glori_dim * 2, 1)

  def forward(self, coarse_patch_tokens, cls_token):
    B = coarse_patch_tokens.shape[0]
    queries = self.disease_queries.expand(B, -1, -1)
    proj_coarse = self.pyramid_proj(coarse_patch_tokens)

    pooled = proj_coarse.mean(dim=1)
    tau = torch.exp(self.temp_mlp(pooled)).unsqueeze(-1)
    queries_scaled = queries / (tau + 1e-6)

    coarse_feat, _ = self.mha_coarse(queries_scaled, proj_coarse, proj_coarse)
    global_feat = (
        self.cls_proj(cls_token).unsqueeze(1).expand(-1, self.num_queries, -1)
    )

    integrated = torch.cat([coarse_feat, global_feat], dim=-1)
    return self.classifier(self.norm(integrated)).squeeze(-1)


def train_eval_glori(tr_idx, ev_idx, lr, epochs=40, batch_size=64):
  torch.manual_seed(42)
  m = CompactGLoRIHead(
      in_dim=1024, glori_dim=768, num_queries=len(TARGET_CLASSES)
  ).to(device)
  opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
  sched = torch.optim.lr_scheduler.CosineAnnealingLR(
      opt, T_max=epochs, eta_min=1e-6
  )
  crit = nn.BCEWithLogitsLoss()

  n_tr = len(tr_idx)
  for epoch in range(epochs):
    m.train()
    perm = torch.randperm(n_tr)
    for i in range(0, n_tr, batch_size):
      b_idx = tr_idx[perm[i : i + batch_size]]
      b_patches = coarse_patches[b_idx].float().to(device)
      b_cls = last_block_cls[b_idx].to(device)
      b_y = Y_all[b_idx].to(device)

      opt.zero_grad()
      crit(m(b_patches, b_cls), b_y).backward()
      opt.step()
    sched.step()

  m.eval()
  preds = []
  with torch.no_grad():
    for i in range(0, len(ev_idx), batch_size):
      b_idx = ev_idx[i : i + batch_size]
      b_patches = coarse_patches[b_idx].float().to(device)
      b_cls = last_block_cls[b_idx].to(device)
      preds.append(torch.sigmoid(m(b_patches, b_cls)).cpu().numpy())
  return np.vstack(preds)


print('\n--- Stage 1: GLoRI Multi-Label LR Search ---')
best_glori_lr, best_val_glori = 1e-4, -1.0
for lr in [2e-5, 5e-5, 1e-4, 2e-4]:
  v_preds = train_eval_glori(
      train_idx, val_idx, lr=lr, epochs=25, batch_size=64
  )
  v_auroc = np.mean([
      roc_auc_score(Y_np[val_idx, c], v_preds[:, c])
      for c in range(len(TARGET_CLASSES))
  ])
  print(f'GLoRI LR {lr:8.1e} | Val Mean AUROC: {v_auroc:.4f}')
  if v_auroc > best_val_glori:
    best_val_glori = v_auroc
    best_glori_lr = lr

print(
    f'\n✓ Selected Best GLoRI LR: {best_glori_lr} (Val Mean AUROC:'
    f' {best_val_glori:.4f})'
)
print(
    f'--- Stage 2: Retraining GLoRI on Train+Val ({len(train_val_idx)} scans) &'
    f' Testing ({len(test_idx)} scans) ---'
)
glori_test_probs = train_eval_glori(
    train_val_idx, test_idx, lr=best_glori_lr, epochs=40, batch_size=64
)

glori_aurocs = [
    roc_auc_score(Y_np[test_idx, c], glori_test_probs[:, c])
    for c in range(len(TARGET_CLASSES))
]
glori_auprcs = [
    average_precision_score(Y_np[test_idx, c], glori_test_probs[:, c])
    for c in range(len(TARGET_CLASSES))
]

mean_glori_auroc = np.mean(glori_aurocs)
mean_glori_auprc = np.mean(glori_auprcs)

print('\n' + '=' * 70)
print('     BENCHMARK 2: GLoRI HEAD (VINDR-CXR)')
print('=' * 70)
for i, c in enumerate(TARGET_CLASSES):
  print(f'{c:22} | AUROC: {glori_aurocs[i]:.4f} | AUPRC: {glori_auprcs[i]:.4f}')
print('-' * 70)
print(
    f'Mean Test AUPRC : {mean_glori_auprc:.4f}  (Table IV Target: 0.341 [0.319 -'
    ' 0.367])'
)
print(
    f'Mean Test AUROC : {mean_glori_auroc:.4f}  (Table IV Target: 0.872 [0.813 -'
    ' 0.898])'
)
print('=' * 70)

Device: cuda


Cloning into '/kaggle/working/CheXFound'...
/kaggle/working/CheXFound/chexfound/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/kaggle/working/CheXFound/chexfound/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/kaggle/working/CheXFound/chexfound/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Searching for VinDr-CXR annotations and images...
✓ Strictly isolated labelled VinDr images: 15000 (Expected: 15,000)
✓ Total Cohort Verified: 15000 scans
✓ CheXFound ViT-Large backbone loaded (0 missing keys)


Extracting VinDr (15,000 Scans): 100%|██████████| 469/469 [1:11:11<00:00,  9.11s/it]


✓ Successfully generated and saved safe cache to /kaggle/working/vindr_safe_cache/

Split Sizes -> Train: 10500 | Val: 1500 | Test: 3000


In [ ]:
import os
import gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

TARGET_CLASSES = [
    'Aortic enlargement',
    'Cardiomegaly',
    'Pulmonary fibrosis',
    'Pleural thickening',
    'Pleural effusion',
    'Pneumothorax'
]

save_dir = '/kaggle/working/vindr_safe_cache'
p_x_linear = os.path.join(save_dir, 'X_linear.pt')
p_coarse = os.path.join(save_dir, 'coarse_patches.pt')
p_cls = os.path.join(save_dir, 'last_block_cls.pt')
p_y = os.path.join(save_dir, 'Y_all.pt')

assert os.path.exists(p_x_linear), "Cache file not found!"

# Load representations
print("Loading cached representations...")
X_linear = torch.load(p_x_linear, map_location='cpu')
Y_all = torch.load(p_y, map_location='cpu')
Y_np = Y_all.numpy()
n_total = len(Y_all)
print(f"✓ Successfully loaded {n_total} scans.")

# Stratified 70:10:20 Partition
indices = np.arange(len(Y_np))
has_disease = (Y_np.sum(axis=1) > 0).astype(int)
train_val_idx, test_idx = train_test_split(indices, test_size=0.20, random_state=42, stratify=has_disease)
train_idx, val_idx = train_test_split(train_val_idx, test_size=0.125, random_state=42, stratify=has_disease[train_val_idx])

print(f"Split Sizes -> Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

# =========================================================================
# BENCHMARK 1: FAST GPU-ACCELERATED LINEAR PROBE (8192-dim)
# =========================================================================
print("\n" + "="*70)
print("     BENCHMARK 1: LINEAR PROBE (VINDR-CXR)")
print("="*70)

# Normalize linear features
X_norm = F.normalize(X_linear, p=2, dim=-1)

def train_linear_probe_gpu(X_tr, y_tr, X_val, y_val, X_te, y_te, C_list=[0.01, 0.1, 1.0, 5.0, 10.0]):
    best_val_auc = -1.0
    best_weight_decay = 1.0
    
    # Grid search for best L2 penalty on Val set
    for C in C_list:
        wd = 1.0 / (C * len(X_tr))
        linear_model = nn.Linear(8192, 1).to(device)
        nn.init.zeros_(linear_model.weight)
        nn.init.zeros_(linear_model.bias)
        
        optimizer = torch.optim.LBFGS(linear_model.parameters(), lr=1.0, max_iter=20, history_size=10, line_search_fn="strong_wolfe")
        criterion = nn.BCEWithLogitsLoss()
        
        X_tr_dev = X_tr.to(device)
        y_tr_dev = y_tr.to(device).unsqueeze(1)
        
        def closure():
            optimizer.zero_grad()
            loss = criterion(linear_model(X_tr_dev), y_tr_dev)
            for p in linear_model.parameters():
                loss += 0.5 * wd * torch.norm(p)**2
            loss.backward()
            return loss

        optimizer.step(closure)
        
        with torch.no_grad():
            val_preds = torch.sigmoid(linear_model(X_val.to(device))).cpu().squeeze(1).numpy()
            val_auc = roc_auc_score(y_val.numpy(), val_preds)
            if val_auc > best_val_auc:
                best_val_auc = val_auc
                best_weight_decay = wd

    # Re-train on Train + Val combined
    X_tr_val = torch.cat([X_tr, X_val], dim=0).to(device)
    y_tr_val = torch.cat([y_tr, y_val], dim=0).to(device).unsqueeze(1)
    
    final_model = nn.Linear(8192, 1).to(device)
    nn.init.zeros_(final_model.weight)
    nn.init.zeros_(final_model.bias)
    optimizer = torch.optim.LBFGS(final_model.parameters(), lr=1.0, max_iter=25, history_size=10, line_search_fn="strong_wolfe")
    criterion = nn.BCEWithLogitsLoss()

    def final_closure():
        optimizer.zero_grad()
        loss = criterion(final_model(X_tr_val), y_tr_val)
        for p in final_model.parameters():
            loss += 0.5 * best_weight_decay * torch.norm(p)**2
        loss.backward()
        return loss

    optimizer.step(final_closure)
    
    with torch.no_grad():
        test_preds = torch.sigmoid(final_model(X_te.to(device))).cpu().squeeze(1).numpy()
        
    auc = roc_auc_score(y_te.numpy(), test_preds)
    ap = average_precision_score(y_te.numpy(), test_preds)
    return auc, ap

lp_aurocs, lp_auprcs = [], []
for c_idx, c_name in enumerate(TARGET_CLASSES):
    auc, ap = train_linear_probe_gpu(
        X_norm[train_idx], Y_all[train_idx, c_idx],
        X_norm[val_idx], Y_all[val_idx, c_idx],
        X_norm[test_idx], Y_all[test_idx, c_idx]
    )
    lp_aurocs.append(auc)
    lp_auprcs.append(ap)
    print(f"{c_name:22} | AUROC: {auc:.4f} | AUPRC: {ap:.4f}")

mean_lp_auroc = np.mean(lp_aurocs)
mean_lp_auprc = np.mean(lp_auprcs)

print("-" * 70)
print(f"Mean Test AUPRC : {mean_lp_auprc:.4f}  (Table IV Target: 0.298 [0.280 - 0.317])")
print(f"Mean Test AUROC : {mean_lp_auroc:.4f}  (Table IV Target: 0.838 [0.776 - 0.866])")
print("=" * 70)

# Free linear probe features from RAM before loading coarse patches
del X_linear, X_norm
gc.collect()

# =========================================================================
# BENCHMARK 2: GLoRI MULTI-QUERY HEAD
# =========================================================================
print("\nLoading GLoRI coarse patches and CLS tokens from cache...")
coarse_patches = torch.load(p_coarse, map_location='cpu')
last_block_cls = torch.load(p_cls, map_location='cpu')

class CompactGLoRIHead(nn.Module):
    def __init__(self, in_dim=1024, glori_dim=768, num_queries=6):
        super().__init__()
        self.num_queries = num_queries
        self.disease_queries = nn.Parameter(torch.randn(1, num_queries, glori_dim) * 0.02)
        self.pyramid_proj = nn.Sequential(
            nn.Linear(in_dim, glori_dim),
            nn.ReLU(),
            nn.LayerNorm(glori_dim)
        )
        self.temp_mlp = nn.Sequential(
            nn.Linear(glori_dim, glori_dim // 2),
            nn.Tanh(),
            nn.Linear(glori_dim // 2, num_queries)
        )
        self.mha_coarse = nn.MultiheadAttention(embed_dim=glori_dim, num_heads=8, batch_first=True)
        self.cls_proj = nn.Sequential(nn.Linear(in_dim, glori_dim), nn.LayerNorm(glori_dim))
        self.norm = nn.LayerNorm(glori_dim * 2)
        self.classifier = nn.Linear(glori_dim * 2, 1)

    def forward(self, coarse_patch_tokens, cls_token):
        B = coarse_patch_tokens.shape[0]
        queries = self.disease_queries.expand(B, -1, -1)
        proj_coarse = self.pyramid_proj(coarse_patch_tokens)
        
        pooled = proj_coarse.mean(dim=1)
        tau = torch.exp(self.temp_mlp(pooled)).unsqueeze(-1)
        queries_scaled = queries / (tau + 1e-6)
        
        coarse_feat, _ = self.mha_coarse(queries_scaled, proj_coarse, proj_coarse)
        global_feat = self.cls_proj(cls_token).unsqueeze(1).expand(-1, self.num_queries, -1)
        
        integrated = torch.cat([coarse_feat, global_feat], dim=-1)
        return self.classifier(self.norm(integrated)).squeeze(-1)

def train_eval_glori(tr_idx, ev_idx, lr, epochs=40, batch_size=64):
    torch.manual_seed(42)
    m = CompactGLoRIHead(in_dim=1024, glori_dim=768, num_queries=len(TARGET_CLASSES)).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    crit = nn.BCEWithLogitsLoss()
    
    n_tr = len(tr_idx)
    for epoch in range(epochs):
        m.train()
        perm = torch.randperm(n_tr)
        for i in range(0, n_tr, batch_size):
            b_idx = tr_idx[perm[i:i + batch_size]]
            b_patches = coarse_patches[b_idx].float().to(device)
            b_cls = last_block_cls[b_idx].to(device)
            b_y = Y_all[b_idx].to(device)
            
            opt.zero_grad()
            crit(m(b_patches, b_cls), b_y).backward()
            opt.step()
        sched.step()
        
    m.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(ev_idx), batch_size):
            b_idx = ev_idx[i:i + batch_size]
            b_patches = coarse_patches[b_idx].float().to(device)
            b_cls = last_block_cls[b_idx].to(device)
            preds.append(torch.sigmoid(m(b_patches, b_cls)).cpu().numpy())
    return np.vstack(preds)

print("\n--- Stage 1: GLoRI Multi-Label LR Search ---")
best_glori_lr, best_val_glori = 1e-4, -1.0
for lr in [2e-5, 5e-5, 1e-4, 2e-4]:
    v_preds = train_eval_glori(train_idx, val_idx, lr=lr, epochs=25, batch_size=64)
    v_auroc = np.mean([roc_auc_score(Y_np[val_idx, c], v_preds[:, c]) for c in range(len(TARGET_CLASSES))])
    print(f"GLoRI LR {lr:8.1e} | Val Mean AUROC: {v_auroc:.4f}")
    if v_auroc > best_val_glori:
        best_val_glori = v_auroc
        best_glori_lr = lr

print(f"\n✓ Selected Best GLoRI LR: {best_glori_lr} (Val Mean AUROC: {best_val_glori:.4f})")
print(f"--- Stage 2: Retraining GLoRI on Train+Val ({len(train_val_idx)} scans) & Testing ({len(test_idx)} scans) ---")
glori_test_probs = train_eval_glori(train_val_idx, test_idx, lr=best_glori_lr, epochs=40, batch_size=64)

glori_aurocs = [roc_auc_score(Y_np[test_idx, c], glori_test_probs[:, c]) for c in range(len(TARGET_CLASSES))]
glori_auprcs = [average_precision_score(Y_np[test_idx, c], glori_test_probs[:, c]) for c in range(len(TARGET_CLASSES))]

mean_glori_auroc = np.mean(glori_aurocs)
mean_glori_auprc = np.mean(glori_auprcs)

print("\n" + "="*70)
print("     BENCHMARK 2: GLoRI HEAD (VINDR-CXR)")
print("="*70)
for i, c in enumerate(TARGET_CLASSES):
    print(f"{c:22} | AUROC: {glori_aurocs[i]:.4f} | AUPRC: {glori_auprcs[i]:.4f}")
print("-" * 70)
print(f"Mean Test AUPRC : {mean_glori_auprc:.4f}  (Table IV Target: 0.341 [0.319 - 0.367])")
print(f"Mean Test AUROC : {mean_glori_auroc:.4f}  (Table IV Target: 0.872 [0.813 - 0.898])")
print("="*70)

Device: cuda
Loading cached representations...
✓ Successfully loaded 15000 scans.
Split Sizes -> Train: 10500 | Val: 1500 | Test: 3000

     BENCHMARK 1: LINEAR PROBE (VINDR-CXR)
Aortic enlargement     | AUROC: 0.9810 | AUPRC: 0.9343
Cardiomegaly           | AUROC: 0.9700 | AUPRC: 0.8553
Pulmonary fibrosis     | AUROC: 0.9425 | AUPRC: 0.6790
Pleural thickening     | AUROC: 0.9418 | AUPRC: 0.6824
Pleural effusion       | AUROC: 0.9615 | AUPRC: 0.7214
Pneumothorax           | AUROC: 0.9135 | AUPRC: 0.3166
----------------------------------------------------------------------
Mean Test AUPRC : 0.6981  (Table IV Target: 0.298 [0.280 - 0.317])
Mean Test AUROC : 0.9517  (Table IV Target: 0.838 [0.776 - 0.866])

Loading GLoRI coarse patches and CLS tokens from cache...

--- Stage 1: GLoRI Multi-Label LR Search ---
GLoRI LR  2.0e-05 | Val Mean AUROC: 0.9342
GLoRI LR  5.0e-05 | Val Mean AUROC: 0.9572
GLoRI LR  1.0e-04 | Val Mean AUROC: 0.9537
GLoRI LR  2.0e-04 | Val Mean AUROC: 0.9579

✓ Select